# 自動評価指標と人手評価の相関を調べる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/13_evaluation/metric_correlation.ipynb)

NLP クックブック 13-10 ／ 評価指標と統計的検定

提案した自動評価指標や LLM による評価が、人間の判断とどれくらい一致しているかを、相関係数で示したい。

In [1]:
import numpy as np
from scipy.stats import kendalltau, pearsonr, spearmanr

rng = np.random.default_rng(0)
human = rng.integers(1, 6, 200).astype(float)                    # 人手評価 (1〜5)
metric_a = human * 15 + rng.normal(0, 12, 200)                   # 人手評価とよく合う指標
metric_b = np.exp(human) + rng.normal(0, 15, 200)                # 単調だが非線形な関係の指標
metric_c = rng.normal(50, 15, 200)                               # 無関係な指標

print(f"{'':10}{'Pearson':>9}{'Spearman':>10}{'Kendall':>9}")
for name, m in [("metric A", metric_a), ("metric B", metric_b), ("metric C", metric_c)]:
    print(f"{name:10}{pearsonr(human, m).statistic:9.3f}{spearmanr(human, m).statistic:10.3f}{kendalltau(human, m).statistic:9.3f}")

            Pearson  Spearman  Kendall
metric A      0.873     0.870    0.726
metric B      0.858     0.902    0.767
metric C      0.047     0.052    0.037


In [2]:
# システム単位の相関: 各システムの平均スコア同士を比べる (文単位より高く出やすい)
n_sys, n_sent = 8, 100
quality = np.linspace(2.5, 4.0, n_sys)                            # システムの真の品質
human_s = quality[:, None] + rng.normal(0, 1.0, (n_sys, n_sent))
metric_s = quality[:, None] * 10 + rng.normal(0, 12, (n_sys, n_sent))
print("文単位:", round(pearsonr(human_s.ravel(), metric_s.ravel()).statistic, 3),
      " システム単位:", round(pearsonr(human_s.mean(1), metric_s.mean(1)).statistic, 3))

# 相関係数の信頼区間 (ブートストラップ)
idx = rng.integers(0, 200, (2000, 200))
boots = [spearmanr(human[i], metric_a[i]).statistic for i in idx]
print("Spearman の 95% 信頼区間:", np.percentile(boots, [2.5, 97.5]).round(3))

文単位: 0.202  システム単位: 0.97
Spearman の 95% 信頼区間: [0.83  0.899]


- Pearson は線形な関係の強さ、Spearman と Kendall は順位の一致を測る。非線形だが単調な関係 (metric B) は Pearson だと過小評価される。
- 同点の多い離散的な評価（1〜5 の評点）には Kendall の τ がよく使われる。
- システム単位の相関は、システム数が少ないと不安定。数個のシステムでの高い相関は当てにならない。
- 相関係数の信頼区間は、評価の組を復元抽出して求めている。考え方はこの章の「ブートストラップ法で評価値の信頼区間を求める」と同じ。
- LLM による評価 (LLM-as-a-judge) は、LLM の評価と分析の章の「手元の LLM に生成文を採点させる (LLM-as-a-judge)」で扱う。そこでも、この相関で採点の信頼性を確かめている。